# Synteny of the RBH hits

For every reciprocal best hit from `1_run_bidirectional_blast.py`, checks whether other hits of the same
reference cluster (SCO or VNZ) in the same genome lie within `N_NEIGHBOURS` genes on either side.

Gene order is taken from the protein `.faa` files (proteins are in genome order), via `shared/synteny.py`.
Positions run continuously through each file, so on draft genomes two genes at the ends of different contigs
can count as neighbours.

In [12]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path("/vol/local/calarass/Projects/ara_comp_genomics")
sys.path.insert(0, str(PROJECT_DIR / "shared"))
from synteny import find_neighbours, get_gene_iterator

RBH_HITS = PROJECT_DIR / "RBH" / "results" / "diamond_reverseBLAST_streptomycetae" / "reciprocal" / "reciprocal_hits.tsv"
FAA_DIR = Path("/vol/local/calarass/Projects/lipid_genomics/faa_files")
N_NEIGHBOURS = 5

## 1. Load the RBH hits

In [13]:
rbh_hits = pd.read_csv(RBH_HITS, sep="\t")
rbh_hits["locus_tag"] = rbh_hits.sseqid.str.split("|").str[1]

print(f"{len(rbh_hits)} hits, {rbh_hits.genome.nunique()} genomes")
rbh_hits.groupby(["ref_group", "qseqid"]).genome.nunique()

885 hits, 157 genomes


ref_group  qseqid        
SCO        SCO2401           130
           SCO2402           130
           SCO2403           129
           SCO2407            95
           SCO2439           125
           SCO2440           132
VNZ        vnz_33170-lacI     43
           vnz_33175-araB     32
           vnz_33180-araA     32
           vnz_33185-araD     24
Name: genome, dtype: int64

## 2. Match genomes to their `.faa` files

By the genome name in the first header, not the file name (file names are not consistent, e.g. no `_protein` suffix).

In [14]:
def genome_of_faa(faa_file: Path) -> str:
    """Genome name from the first header, brackets dropped to match the genome column of reciprocal_hits.tsv."""
    with open(faa_file) as f:
        header = f.readline()
    return header.lstrip(">").split("|")[0].replace("[", "").replace("]", "")


FAA_FILES = {genome_of_faa(path): path for path in sorted(FAA_DIR.glob("*.faa"))}

missing = sorted(set(rbh_hits.genome) - set(FAA_FILES))
if missing:
    print("WARNING: no .faa file for:", missing)

## 3. Neighbouring hits

Per reference group (SCO / VNZ) and genome: for each hit, which other hits of the same group are among its
`N_NEIGHBOURS` genes to the left and right.

In [15]:
rows = []
for genome, genome_hits in rbh_hits.groupby("genome"):
    gene_positions = get_gene_iterator(FAA_FILES[genome])
    for ref_group, group_hits in genome_hits.groupby("ref_group"):
        hit_genes = dict(zip(group_hits.locus_tag, group_hits.qseqid))
        for hit in group_hits.itertuples():
            neighbours = find_neighbours(hit.locus_tag, gene_positions, N_NEIGHBOURS)
            neighbour_hits = [f"{hit_genes[n]}:{n}" for n in neighbours if n in hit_genes]
            rows.append({
                "ref_group": ref_group,
                "genome": genome,
                "qseqid": hit.qseqid,
                "locus_tag": hit.locus_tag,
                "neighbour_hits": ", ".join(neighbour_hits),
                "has_neighbours": "yes" if neighbour_hits else "no",
            })

synteny_table = pd.DataFrame(rows).sort_values(["ref_group", "genome", "qseqid"]).reset_index(drop=True)
synteny_table

,ref_group,genome,qseqid,locus_tag,neighbour_hits,has_neighbours
0,SCO,Actinacidiphila_glaucinigra,SCO2401,OIE69_RS26950,"SCO2403:OIE69_RS26940, SCO2402:OIE69_RS26945",yes
1,SCO,Actinacidiphila_glaucinigra,SCO2402,OIE69_RS26945,"SCO2407:OIE69_RS26920, SCO2403:OIE69_RS26940, ...",yes
2,SCO,Actinacidiphila_glaucinigra,SCO2403,OIE69_RS26940,"SCO2407:OIE69_RS26920, SCO2402:OIE69_RS26945, ...",yes
3,SCO,Actinacidiphila_glaucinigra,SCO2407,OIE69_RS26920,"SCO2403:OIE69_RS26940, SCO2402:OIE69_RS26945",yes
4,SCO,Actinacidiphila_glaucinigra,SCO2439,OIE69_RS26865,SCO2440:OIE69_RS26860,yes
...,...,...,...,...,...,...
880,VNZ,Streptomyces_venezuelae_ATCC_10712,vnz_33185-araD,DEJ43_RS33670,"vnz_33170-lacI:DEJ43_RS33655, vnz_33175-araB:D...",yes
881,VNZ,Streptomyces_vietnamensis,vnz_33170-lacI,SVTN_RS36800,"vnz_33175-araB:SVTN_RS36805, vnz_33180-araA:SV...",yes
882,VNZ,Streptomyces_vietnamensis,vnz_33175-araB,SVTN_RS36805,"vnz_33170-lacI:SVTN_RS36800, vnz_33180-araA:SV...",yes
883,VNZ,Streptomyces_vietnamensis,vnz_33180-araA,SVTN_RS36810,"vnz_33170-lacI:SVTN_RS36800, vnz_33175-araB:SV...",yes


In [16]:
synteny_table.loc[synteny_table.has_neighbours == "no"].groupby(["ref_group", "qseqid"]).genome.nunique().sort_values(ascending=False).head(10)

ref_group  qseqid        
VNZ        vnz_33170-lacI    22
SCO        SCO2439            6
           SCO2407            6
           SCO2402            2
           SCO2440            2
VNZ        vnz_33175-araB     1
           vnz_33185-araD     1
Name: genome, dtype: int64

### 3b. SCO2401 / SCO2402 / SCO2403 trio

In *S. coelicolor* SCO2401-SCO2403 sit next to each other. Per genome and query gene: is the gene directly to the
left or right (`TRIO_WINDOW` genes) of the hit itself a hit of another trio gene?

`trio_summary` then gives one row per genome: `trio` = all three genes in one adjacent block, `pair` = at most two
of them adjacent, `none` = hits present but not adjacent. Tandem copies of the same gene next to each other do not count.

In [17]:
TRIO = ["SCO2401", "SCO2402", "SCO2403"]
TRIO_WINDOW = 1  # direct left/right neighbour only

trio_hits = rbh_hits[rbh_hits.qseqid.isin(TRIO)]

rows = []
for genome, genome_hits in trio_hits.groupby("genome"):
    gene_positions = get_gene_iterator(FAA_FILES[genome])
    hit_genes = dict(zip(genome_hits.locus_tag, genome_hits.qseqid))
    for hit in genome_hits.itertuples():
        neighbours = find_neighbours(hit.locus_tag, gene_positions, TRIO_WINDOW)
        # another trio gene next to it - a neighbouring copy of the same gene is not counted
        trio_neighbours = [f"{hit_genes[n]}:{n}" for n in neighbours if n in hit_genes and hit_genes[n] != hit.qseqid]
        rows.append({
            "genome": genome,
            "qseqid": hit.qseqid,
            "locus_tag": hit.locus_tag,
            "position": gene_positions[hit.locus_tag],
            "trio_neighbours": ", ".join(trio_neighbours),
            "has_any_trio_neighbour": "yes" if trio_neighbours else "no",
        })

trio_table = pd.DataFrame(rows).sort_values(["genome", "position"]).reset_index(drop=True)
trio_table

,genome,qseqid,locus_tag,position,trio_neighbours,has_any_trio_neighbour
0,Actinacidiphila_glaucinigra,SCO2403,OIE69_RS26940,5191,SCO2402:OIE69_RS26945,yes
1,Actinacidiphila_glaucinigra,SCO2402,OIE69_RS26945,5192,"SCO2403:OIE69_RS26940, SCO2401:OIE69_RS26950",yes
2,Actinacidiphila_glaucinigra,SCO2401,OIE69_RS26950,5193,SCO2402:OIE69_RS26945,yes
3,Kitasatospora_papulosa,SCO2403,OG362_RS22095,4297,SCO2402:OG362_RS22100,yes
4,Kitasatospora_papulosa,SCO2402,OG362_RS22100,4298,"SCO2403:OG362_RS22095, SCO2401:OG362_RS22105",yes
...,...,...,...,...,...,...
386,Streptomyces_yatensis,SCO2402,J8403_RS29520,5748,"SCO2403:J8403_RS29515, SCO2401:J8403_RS29525",yes
387,Streptomyces_yatensis,SCO2401,J8403_RS29525,5749,SCO2402:J8403_RS29520,yes
388,Streptomyces_zaomyceticus,SCO2403,OG567_RS33905,6619,,no
389,Streptomyces_zaomyceticus,SCO2402,OG567_RS34790,6793,SCO2401:OG567_RS34795,yes


In [19]:
trio_table.groupby("genome").agg(
    n_hits=("qseqid", "size"),
    genes=("qseqid", ", ".join),
    locus_tags=("locus_tag", ", ".join),
    positions=("position", list),
    with_trio_neighbour=("has_any_trio_neighbour", lambda s: (s == "yes").sum()),
).head(20)


,n_hits,genes,locus_tags,positions,with_trio_neighbour
genome,,,,,
Actinacidiphila_glaucinigra,3,"SCO2403, SCO2402, SCO2401","OIE69_RS26940, OIE69_RS26945, OIE69_RS26950","[5191, 5192, 5193]",3
Kitasatospora_papulosa,3,"SCO2403, SCO2402, SCO2401","OG362_RS22095, OG362_RS22100, OG362_RS22105","[4297, 4298, 4299]",3
Peterkaempfera_sp._SMS_1(5)a,2,"SCO2403, SCO2402","AB0761_RS30805, AB0761_RS30810","[5790, 5791]",2
Streptomyces_achromogenes,3,"SCO2401, SCO2402, SCO2403","OG350_RS12710, OG350_RS12715, OG350_RS12720","[2465, 2466, 2467]",3
Streptomyces_akebiae,3,"SCO2403, SCO2402, SCO2401","K1J60_RS31245, K1J60_RS31250, K1J60_RS31255","[6065, 6066, 6067]",3
Streptomyces_albidocamelliae,3,"SCO2401, SCO2402, SCO2403","N8I86_RS13315, N8I86_RS13320, N8I86_RS13325","[2521, 2522, 2523]",3
Streptomyces_alboniger,3,"SCO2403, SCO2402, SCO2401","CP975_RS30020, CP975_RS30025, CP975_RS30030","[5824, 5825, 5826]",3
Streptomyces_alfalfae,3,"SCO2403, SCO2402, SCO2401","H9W91_RS31130, H9W91_RS31135, H9W91_RS31140","[6051, 6052, 6053]",3
Streptomyces_althioticus,3,"SCO2403, SCO2402, SCO2401","OG872_RS23315, OG872_RS23320, OG872_RS23325","[4520, 4521, 4522]",3


In [20]:
def adjacent_blocks(genome_hits: pd.DataFrame) -> list[set[str]]:
    """Split a genome's trio hits (sorted by position) into runs of genes at most TRIO_WINDOW apart;
    return the set of trio genes in each run."""
    blocks = []
    previous = None
    for hit in genome_hits.itertuples():
        if previous is None or hit.position - previous > TRIO_WINDOW:
            blocks.append(set())
        blocks[-1].add(hit.qseqid)
        previous = hit.position
    return blocks


rows = []
for genome, genome_hits in trio_table.groupby("genome"):
    largest = max(adjacent_blocks(genome_hits), key=len)
    rows.append({
        "genome": genome,
        "trio_genes_found": ", ".join(sorted(genome_hits.qseqid.unique())),
        "largest_block": ", ".join(sorted(largest)),
        "neighbourhood": {3: "trio", 2: "pair"}.get(len(largest), "none"),
    })

trio_summary = pd.DataFrame(rows)
print(f"{len(trio_summary)} genomes with at least one SCO2401-2403 hit")
trio_summary.groupby(["trio_genes_found", "neighbourhood"]).size()

132 genomes with at least one SCO2401-2403 hit


trio_genes_found           neighbourhood
SCO2401                    none               2
SCO2401, SCO2402, SCO2403  pair               1
                           trio             127
SCO2402                    none               1
SCO2402, SCO2403           pair               1
dtype: int64

In [21]:
# genomes where the three genes are not all in one block
trio_summary[trio_summary.neighbourhood != "trio"].sort_values(["neighbourhood", "genome"])

,genome,trio_genes_found,largest_block,neighbourhood
53,Streptomyces_gobiensis,SCO2402,SCO2402,none
121,Streptomyces_venezuelae,SCO2401,SCO2401,none
122,Streptomyces_venezuelae_ATCC_10712,SCO2401,SCO2401,none
2,Peterkaempfera_sp._SMS_1(5)a,"SCO2402, SCO2403","SCO2402, SCO2403",pair
131,Streptomyces_zaomyceticus,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402",pair


In [22]:
trio_summary.head(29)

,genome,trio_genes_found,largest_block,neighbourhood
0,Actinacidiphila_glaucinigra,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio
1,Kitasatospora_papulosa,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio
2,Peterkaempfera_sp._SMS_1(5)a,"SCO2402, SCO2403","SCO2402, SCO2403",pair
3,Streptomyces_achromogenes,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio
4,Streptomyces_akebiae,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio
5,Streptomyces_albidocamelliae,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio
6,Streptomyces_alboniger,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio
7,Streptomyces_alfalfae,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio
8,Streptomyces_althioticus,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio
9,Streptomyces_ambofaciens_ATCC_23877,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio


In [23]:
trio_summary[trio_summary.genome.str.contains("griseorubiginosus")]

,genome,trio_genes_found,largest_block,neighbourhood
58,Streptomyces_griseorubiginosus,"SCO2401, SCO2402, SCO2403","SCO2401, SCO2402, SCO2403",trio


## 4. Investigate hits without neighbours

`isolated_hits`: for every hit with `has_neighbours == "no"`, the nearest other hit of the same reference group
in that genome and how many genes away it is (NaN = the only hit of its group in that genome).

`show_neighbourhood` lists the `WINDOW` genes on each side of a hit with their full description from the `.faa`
header; `rbh_hit` marks genes that are themselves RBH hits (of either group).

In [41]:
WINDOW = 10


def read_descriptions(faa_file: Path) -> dict[str, str]:
    """Return {locus_tag: description} from the '>Genome|LOCUS_TAG|description|accession' headers."""
    descriptions = {}
    with open(faa_file) as f:
        for line in f:
            if line.startswith(">"):
                fields = line.split("|")
                descriptions[fields[1]] = fields[2]
    return descriptions


def show_neighbourhood(genome: str, locus_tag: str, window: int = WINDOW) -> pd.DataFrame:
    """The window genes on each side of locus_tag, in genome order, with descriptions and RBH hits marked."""
    gene_positions = get_gene_iterator(FAA_FILES[genome])
    descriptions = read_descriptions(FAA_FILES[genome])
    genome_hits = synteny_table[synteny_table.genome == genome]
    hit_genes = dict(zip(genome_hits.locus_tag, genome_hits.qseqid))

    genes = sorted(find_neighbours(locus_tag, gene_positions, window) + [locus_tag], key=gene_positions.get)
    return pd.DataFrame({
        "offset": [gene_positions[g] - gene_positions[locus_tag] for g in genes],
        "locus_tag": genes,
        "description": [descriptions[g] for g in genes],
        "rbh_hit": [hit_genes.get(g, "") for g in genes],
    })


rows = []
for genome, genome_isolated in synteny_table[synteny_table.has_neighbours == "no"].groupby("genome"):
    gene_positions = get_gene_iterator(FAA_FILES[genome])
    genome_hits = synteny_table[synteny_table.genome == genome]
    for hit in genome_isolated.itertuples():
        others = genome_hits[(genome_hits.ref_group == hit.ref_group) & (genome_hits.locus_tag != hit.locus_tag)]
        distances = {f"{o.qseqid}:{o.locus_tag}": abs(gene_positions[o.locus_tag] - gene_positions[hit.locus_tag])
                     for o in others.itertuples()}
        nearest = min(distances, key=distances.get) if distances else None
        rows.append({
            "ref_group": hit.ref_group,
            "genome": genome,
            "qseqid": hit.qseqid,
            "locus_tag": hit.locus_tag,
            "other_group_hits": len(others),
            "nearest_hit": nearest,
            "genes_away": distances.get(nearest),
        })

isolated_hits = pd.DataFrame(rows).sort_values(["ref_group", "qseqid", "genes_away"]).reset_index(drop=True)
isolated_hits.groupby(["ref_group", "qseqid"]).size()

ref_group  qseqid        
SCO        SCO2402            2
           SCO2407           23
           SCO2439            5
           SCO2440            2
VNZ        vnz_33170-lacI    25
           vnz_33175-araB     2
           vnz_33180-araA     1
           vnz_33185-araD     2
dtype: int64

### SCO2407

In [42]:
sco2407 = isolated_hits[isolated_hits.qseqid.isin(["SCO2407"])]
sco2407

,ref_group,genome,qseqid,locus_tag,other_group_hits,nearest_hit,genes_away
2,SCO,Streptomyces_seoulensis,SCO2407,D0Z67_RS08555,5,SCO2401:D0Z67_RS08585,6.0
3,SCO,Streptomyces_dengpaensis,SCO2407,C4B68_RS27745,5,SCO2401:C4B68_RS27695,8.0
4,SCO,Streptomyces_cynarae,SCO2407,N8I84_RS13275,5,SCO2401:N8I84_RS13325,10.0
5,SCO,Streptomyces_longhuiensis,SCO2407,LGI35_RS16010,5,SCO2401:LGI35_RS16060,10.0
6,SCO,Streptomyces_bobili,SCO2407,OHT53_RS28210,5,SCO2401:OHT53_RS28135,14.0
7,SCO,Streptomyces_phaeoluteigriseus,SCO2407,NFX46_RS32755,5,SCO2401:NFX46_RS32835,15.0
8,SCO,Streptomyces_ferrugineus,SCO2407,IM697_RS09605,5,SCO2401:IM697_RS09525,16.0
9,SCO,Streptomyces_collinus,SCO2407,RFN52_RS12085,5,SCO2401:RFN52_RS12180,19.0
10,SCO,Streptomyces_canus,SCO2407,OHT68_RS32760,5,SCO2401:OHT68_RS32655,21.0
11,SCO,Streptomyces_cyaneochromogenes,SCO2407,EJ357_RS14015,5,SCO2401:EJ357_RS14170,28.0


In [43]:
for hit in sco2407.itertuples():
    print(f"{hit.genome}  {hit.locus_tag}  (nearest: {hit.nearest_hit}, {hit.genes_away} genes away)")
    display(show_neighbourhood(hit.genome, hit.locus_tag))

Streptomyces_seoulensis  D0Z67_RS08555  (nearest: SCO2401:D0Z67_RS08585, 6.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,D0Z67_RS08505,GNAT family N-acetyltransferase,
1,-9,D0Z67_RS08510,serine hydrolase domain-containing protein,
2,-8,D0Z67_RS08515,pirin family protein,
3,-7,D0Z67_RS08520,PucR family transcriptional regulator,
4,-6,D0Z67_RS08525,ACP S-malonyltransferase,
5,-5,D0Z67_RS08530,ketoacyl-ACP synthase III,
6,-4,D0Z67_RS08535,acyl carrier protein,
7,-3,D0Z67_RS08540,beta-ketoacyl-ACP synthase II,
8,-2,D0Z67_RS08545,DUF3145 domain-containing protein,
9,-1,D0Z67_RS08550,SGNH/GDSL hydrolase family protein,


Streptomyces_dengpaensis  C4B68_RS27745  (nearest: SCO2401:C4B68_RS27695, 8.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,C4B68_RS27685,hypothetical protein,SCO2403
1,-9,C4B68_RS27690,zinc-dependent alcohol dehydrogenase,SCO2402
2,-8,C4B68_RS27695,mandelate racemase/muconate lactonizing enzyme...,SCO2401
3,-7,C4B68_RS42115,SCO2400 family protein,
4,-6,C4B68_RS27715,hypothetical protein,
5,-5,C4B68_RS27720,M14 family zinc carboxypeptidase,
6,-4,C4B68_RS27725,NADP-dependent oxidoreductase,
7,-3,C4B68_RS27730,MarR family winged helix-turn-helix transcript...,
8,-2,C4B68_RS27735,EI24 domain-containing protein,
9,-1,C4B68_RS27740,pyroglutamyl peptidase,


Streptomyces_cynarae  N8I84_RS13275  (nearest: SCO2401:N8I84_RS13325, 10.0 genes away)


...

,offset,locus_tag,description,rbh_hit
0,-10,GO605_RS29880,serine/threonine-protein kinase,
1,-9,GO605_RS29885,pirin family protein,
2,-8,GO605_RS29890,fatty acid biosynthesis transcriptional regula...,
3,-7,GO605_RS29895,ACP S-malonyltransferase,
4,-6,GO605_RS29900,ketoacyl-ACP synthase III,
5,-5,GO605_RS29905,acyl carrier protein,
6,-4,GO605_RS29910,beta-ketoacyl-ACP synthase II,
7,-3,GO605_RS29915,DUF3145 domain-containing protein,
8,-2,GO605_RS29920,SGNH/GDSL hydrolase family protein,
9,-1,GO605_RS29925,SRPBCC family protein,


Streptomyces_nodosus  CP978_RS11535  (nearest: None, nan genes away)


,offset,locus_tag,description,rbh_hit
0,-10,CP978_RS11485,hypothetical protein,
1,-9,CP978_RS11490,serine hydrolase domain-containing protein,
2,-8,CP978_RS11495,pirin family protein,
3,-7,CP978_RS11500,PucR family transcriptional regulator,
4,-6,CP978_RS11505,ACP S-malonyltransferase,
5,-5,CP978_RS11510,ketoacyl-ACP synthase III,
6,-4,CP978_RS11515,acyl carrier protein,
7,-3,CP978_RS11520,beta-ketoacyl-ACP synthase II,
8,-2,CP978_RS11525,DUF3145 domain-containing protein,
9,-1,CP978_RS11530,SGNH/GDSL hydrolase family protein,


Streptomyces_tubercidicus  STRTU_RS24690  (nearest: None, nan genes away)


,offset,locus_tag,description,rbh_hit
0,-10,STRTU_RS24630,organic hydroperoxide resistance protein,
1,-9,STRTU_RS24635,hypothetical protein,
2,-8,STRTU_RS24640,LacI family DNA-binding transcriptional regulator,
3,-7,STRTU_RS24645,sugar ABC transporter ATP-binding protein,
4,-6,STRTU_RS24655,ribokinase,
5,-5,STRTU_RS35990,nitrilase-related carbon-nitrogen hydrolase,
6,-4,STRTU_RS24670,pyroglutamyl peptidase,
7,-3,STRTU_RS24675,TetR/AcrR family transcriptional regulator,
8,-2,STRTU_RS24680,beta-glucosidase family protein,
9,-1,STRTU_RS24685,DUF4097 family beta strand repeat-containing p...,


### SCO2439

In [44]:
sco2439 = isolated_hits[isolated_hits.qseqid.isin(["SCO2439"])]
sco2439

,ref_group,genome,qseqid,locus_tag,other_group_hits,nearest_hit,genes_away
25,SCO,Streptomyces_niveus,SCO2439,OG275_RS01590,4,SCO2440:OG275_RS00925,127.0
26,SCO,Streptomyces_rosealbus,SCO2439,ACRAKG_RS14275,5,SCO2407:ACRAKG_RS06195,1585.0
27,SCO,Streptomyces_durmitorensis,SCO2439,M4V62_RS28785,5,SCO2407:M4V62_RS37490,1706.0
28,SCO,Streptomyces_boninensis,SCO2439,ACNNMR_RS12515,5,SCO2440:ACNNMR_RS23265,2080.0
29,SCO,Streptomyces_halobius,SCO2439,K9S39_RS40770,0,None,NaN


In [45]:
for hit in sco2439.itertuples():
    print(f"{hit.genome}  {hit.locus_tag}  (nearest: {hit.nearest_hit}, {hit.genes_away} genes away)")
    display(show_neighbourhood(hit.genome, hit.locus_tag))

Streptomyces_niveus  OG275_RS01590  (nearest: SCO2440:OG275_RS00925, 127.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,OG275_RS01540,hypothetical protein,
1,-9,OG275_RS01545,hypothetical protein,
2,-8,OG275_RS01550,cold-shock protein,
3,-7,OG275_RS01555,DEAD/DEAH box helicase,
4,-6,OG275_RS01560,SCO5918 family protein,
5,-5,OG275_RS01565,hypothetical protein,
6,-4,OG275_RS01570,endonuclease,
7,-3,OG275_RS01575,hypothetical protein,
8,-2,OG275_RS01580,SpoIIE family protein phosphatase,
9,-1,OG275_RS01585,DUF6351 family protein,


Streptomyces_rosealbus  ACRAKG_RS14275  (nearest: SCO2407:ACRAKG_RS06195, 1585.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,ACRAKG_RS14225,LysR family transcriptional regulator,
1,-9,ACRAKG_RS14230,DUF5713 family protein,
2,-8,ACRAKG_RS14235,pyroglutamyl-peptidase I family protein,
3,-7,ACRAKG_RS14240,EI24 domain-containing protein,
4,-6,ACRAKG_RS14245,M14 family zinc carboxypeptidase,
5,-5,ACRAKG_RS14250,serine hydrolase domain-containing protein,
6,-4,ACRAKG_RS14255,organic hydroperoxide resistance protein,
7,-3,ACRAKG_RS14260,NADP-dependent oxidoreductase,
8,-2,ACRAKG_RS14265,MarR family winged helix-turn-helix transcript...,
9,-1,ACRAKG_RS14270,SCO2400 family protein,


Streptomyces_durmitorensis  M4V62_RS28785  (nearest: SCO2407:M4V62_RS37490, 1706.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,M4V62_RS28725,FkbM family methyltransferase,
1,-9,M4V62_RS28730,Trm112 family protein,
2,-8,M4V62_RS28735,hypothetical protein,
3,-7,M4V62_RS28740,class I SAM-dependent methyltransferase,
4,-6,M4V62_RS28755,DUF3367 domain-containing protein,
5,-5,M4V62_RS28760,class I SAM-dependent methyltransferase,
6,-4,M4V62_RS28765,glycosyltransferase family 4 protein,
7,-3,M4V62_RS28770,DUF3068 domain-containing protein,
8,-2,M4V62_RS28775,hypothetical protein,
9,-1,M4V62_RS28780,helix-turn-helix domain-containing protein,


Streptomyces_boninensis  ACNNMR_RS12515  (nearest: SCO2440:ACNNMR_RS23265, 2080.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,ACNNMR_RS12465,DUF3145 domain-containing protein,
1,-9,ACNNMR_RS12470,GDSL-type esterase/lipase family protein,
2,-8,ACNNMR_RS12475,aldose epimerase family protein,
3,-7,ACNNMR_RS12480,glycoside hydrolase family 172 protein,
4,-6,ACNNMR_RS12485,GNAT family N-acetyltransferase,
5,-5,ACNNMR_RS12490,ATP-binding protein,
6,-4,ACNNMR_RS12495,organic hydroperoxide resistance protein,
7,-3,ACNNMR_RS12500,NADP-dependent oxidoreductase,
8,-2,ACNNMR_RS12505,MarR family winged helix-turn-helix transcript...,
9,-1,ACNNMR_RS12510,SpoIIE family protein phosphatase,


Streptomyces_halobius  K9S39_RS40770  (nearest: None, nan genes away)


,offset,locus_tag,description,rbh_hit
0,-10,K9S39_RS40710,roadblock/LC7 domain-containing protein,
1,-9,K9S39_RS40715,ATP-binding protein,
2,-8,K9S39_RS40720,substrate-binding domain-containing protein,
3,-7,K9S39_RS40725,methyltransferase domain-containing protein,
4,-6,K9S39_RS40730,aminotransferase class V-fold PLP-dependent en...,
5,-5,K9S39_RS40735,ArsR/SmtB family transcription factor,
6,-4,K9S39_RS40740,SHOCT domain-containing protein,
7,-3,K9S39_RS40745,hypothetical protein,
8,-2,K9S39_RS40760,PepSY domain-containing protein,
9,-1,K9S39_RS40765,IclR family transcriptional regulator domain-c...,


### VNZ araB and araD

In [46]:
vnz_ara = isolated_hits[isolated_hits.qseqid.isin(["vnz_33175-araB", "vnz_33185-araD"])]
vnz_ara

,ref_group,genome,qseqid,locus_tag,other_group_hits,nearest_hit,genes_away
57,VNZ,Streptomyces_erythrochromogenes,vnz_33175-araB,OHA91_RS05465,3,vnz_33180-araA:OHA91_RS05365,16.0
58,VNZ,Streptomyces_iranensis,vnz_33175-araB,RY074_RS00855,0,None,NaN
60,VNZ,Streptomyces_canus,vnz_33185-araD,OHT68_RS00520,3,vnz_33175-araB:OHT68_RS02535,356.0
61,VNZ,Streptomyces_ipomoeae,vnz_33185-araD,ACNJZD_RS24120,1,vnz_33170-lacI:ACNJZD_RS14055,1930.0


In [47]:
for hit in vnz_ara.itertuples():
    print(f"{hit.genome}  {hit.locus_tag}  (nearest: {hit.nearest_hit}, {hit.genes_away} genes away)")
    display(show_neighbourhood(hit.genome, hit.locus_tag))

Streptomyces_erythrochromogenes  OHA91_RS05465  (nearest: vnz_33180-araA:OHA91_RS05365, 16.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,OHA91_RS05395,hypothetical protein,
1,-9,OHA91_RS05400,ATP-binding protein,
2,-8,OHA91_RS05405,hypothetical protein,
3,-7,OHA91_RS05410,hypothetical protein,
4,-6,OHA91_RS05415,ROK family protein,
5,-5,OHA91_RS05420,xylulokinase,
6,-4,OHA91_RS05425,xylose isomerase,
7,-3,OHA91_RS05435,GH1 family beta-glucosidase,
8,-2,OHA91_RS05450,aldose epimerase family protein,
9,-1,OHA91_RS05455,hypothetical protein,


Streptomyces_iranensis  RY074_RS00855  (nearest: None, nan genes away)


,offset,locus_tag,description,rbh_hit
0,-10,RY074_RS50075,acyl carrier protein,
1,-9,RY074_RS00800,cytochrome P450,
2,-8,RY074_RS00805,SDR family NAD(P)-dependent oxidoreductase,
3,-7,RY074_RS00810,hypothetical protein,
4,-6,RY074_RS00815,hypothetical protein,
5,-5,RY074_RS00820,transposase,
6,-4,RY074_RS00835,SRPBCC domain-containing protein,
7,-3,RY074_RS00840,IS701 family transposase,
8,-2,RY074_RS00845,beta-galactosidase,
9,-1,RY074_RS00850,L-ribulose-5-phosphate 4-epimerase,


Streptomyces_canus  OHT68_RS00520  (nearest: vnz_33175-araB:OHT68_RS02535, 356.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,OHT68_RS00455,DUF6192 family protein,
1,-9,OHT68_RS00460,hypothetical protein,
2,-8,OHT68_RS00465,hypothetical protein,
3,-7,OHT68_RS00475,Type 1 glutamine amidotransferase-like domain-...,
4,-6,OHT68_RS00480,NF041680 family putative transposase,
5,-5,OHT68_RS00485,glutaredoxin domain-containing protein,
6,-4,OHT68_RS00490,alpha/beta fold hydrolase,
7,-3,OHT68_RS00500,NF041680 family putative transposase,
8,-2,OHT68_RS00505,transposase,
9,-1,OHT68_RS00510,hypothetical protein,


Streptomyces_ipomoeae  ACNJZD_RS24120  (nearest: vnz_33170-lacI:ACNJZD_RS14055, 1930.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,ACNJZD_RS24060,cold-shock protein,
1,-9,ACNJZD_RS24065,MerR family transcriptional regulator,
2,-8,ACNJZD_RS24075,hypothetical protein,
3,-7,ACNJZD_RS24080,sigma-70 family RNA polymerase sigma factor,
4,-6,ACNJZD_RS24085,multicopper oxidase family protein,
5,-5,ACNJZD_RS24090,PA domain-containing protein,
6,-4,ACNJZD_RS24095,extracellular solute-binding protein,
7,-3,ACNJZD_RS24100,carbohydrate ABC transporter permease,
8,-2,ACNJZD_RS24105,IS200/IS605 family transposase,
9,-1,ACNJZD_RS24110,RNA-guided endonuclease InsQ/TnpB family protein,


# what can be removed
ACNJZD_RS24120 reason: the only protein in the pathway
